# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/prarthanamahesh21-hub/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


I use five page-level features available before the prediction point: content age, word count, Google Search Console impressions over the prior 90 days, Google Search Console clicks over the prior 90 days, and GA4 pageviews over the prior 90 days. The feature frame is built for the March 2026 decision period. Missing numeric values are handled with median imputation during modeling.


In [3]:
import pandas as pd
import duckdb
from datasets import load_dataset
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

dim_content = load_dataset(
    "FlyRank/internship-warehouse",
    "dim_content",
    token=HF_TOKEN
)

fact_performance = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    token=HF_TOKEN
)

content = dim_content["train"]
performance = fact_performance["train"]

con = duckdb.connect()

con.register("dim_content", content._data.table)
con.register(
    "fact_content_daily_performance",
    performance._data.table
)

features_march = con.sql("""
WITH performance_90d AS (
    SELECT
        content_hash_id,

        SUM(
            CASE
                WHEN gsc_data_available IS TRUE
                THEN COALESCE(gsc_impressions, 0)
                ELSE 0
            END
        ) AS gsc_impressions_90d,

        SUM(
            CASE
                WHEN gsc_data_available IS TRUE
                THEN COALESCE(gsc_clicks, 0)
                ELSE 0
            END
        ) AS gsc_clicks_90d,

        SUM(
            CASE
                WHEN ga4_data_available IS TRUE
                THEN COALESCE(ga4_pageviews, 0)
                ELSE 0
            END
        ) AS ga4_pageviews_90d

    FROM fact_content_daily_performance

    WHERE report_date >= DATE '2025-12-01'
      AND report_date < DATE '2026-03-01'

    GROUP BY content_hash_id
)

SELECT
    c.content_hash_id,

    DATE_DIFF(
        'day',
        CAST(c.content_created_date AS DATE),
        DATE '2026-03-01'
    ) AS content_age_days,

    c.word_count,

    COALESCE(p.gsc_impressions_90d, 0) AS gsc_impressions_90d,
    COALESCE(p.gsc_clicks_90d, 0) AS gsc_clicks_90d,
    COALESCE(p.ga4_pageviews_90d, 0) AS ga4_pageviews_90d

FROM dim_content c

LEFT JOIN performance_90d p
    ON c.content_hash_id = p.content_hash_id

WHERE c.content_created_date < DATE '2026-03-01'
  AND c.is_published IS TRUE
  AND c.is_deleted IS FALSE
""").df()

feature_cols = [
    "content_age_days",
    "word_count",
    "gsc_impressions_90d",
    "gsc_clicks_90d",
    "ga4_pageviews_90d"
]

X_w03 = features_march[feature_cols].copy()

print("Feature vector shape:", X_w03.shape)
print("Features:")
print(feature_cols)

print("\nMissing values:")
print(X_w03.isna().sum())

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

dim_content.parquet: reconstructing file:   0%|          |  0.00B / 19.6MB            

dim_content.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/519606 [00:00<?, ? examples/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

...

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/78835655 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/39 [00:00<?, ?it/s]

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature vector shape: (303321, 5)
Features:
['content_age_days', 'word_count', 'gsc_impressions_90d', 'gsc_clicks_90d', 'ga4_pageviews_90d']

Missing values:
content_age_days            0
word_count             106638
gsc_impressions_90d         0
gsc_clicks_90d              0
ga4_pageviews_90d           0
dtype: int64


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


* `content_age_days`: number of days since the page was created, measured as of March 1, 2026. It is available before prediction.
* `word_count`: page word count. Missing values are retained in the feature frame and handled by median imputation during modeling. It is available before prediction.
* `gsc_impressions_90d`: Google Search Console impressions observed during the prior 90 days. Missing performance observations are represented as zero where the source indicates no available data. It is available before prediction.
* `gsc_clicks_90d`: Google Search Console clicks observed during the prior 90 days. Missing performance observations are represented as zero where the source indicates no available data. It is available before prediction.
* `ga4_pageviews_90d`: GA4 pageviews observed during the prior 90 days. Missing performance observations are represented as zero where the source indicates no available data. It is available before prediction.

No categorical feature is used in the final model feature vector.


In [4]:
print("Feature availability and missingness:")
print(X_w03.isna().sum())

print("\nFeature dtypes:")
print(X_w03.dtypes)

print("\nNumber of categorical features:",
      sum(X_w03.dtypes == "object"))

Feature availability and missingness:
content_age_days            0
word_count             106638
gsc_impressions_90d         0
gsc_clicks_90d              0
ga4_pageviews_90d           0
dtype: int64

Feature dtypes:
content_age_days         int64
word_count               Int64
gsc_impressions_90d    float64
gsc_clicks_90d         float64
ga4_pageviews_90d      float64
dtype: object

Number of categorical features: 0


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

I checked the feature vector for direct target or outcome-derived fields, future-period measurements, and action/refresh indicators. The April 2026 outcome is stored separately as the target and is not included in the feature vector. The five model features are calculated from page attributes and performance observed before March 1, 2026. Therefore, the future April outcome is not used as an input to the model.


In [5]:
target_like = [
    col for col in X_w03.columns
    if any(word in col.lower()
           for word in ["label", "target", "future", "outcome",
                        "refresh", "action", "positive"])
]

future_like = [
    col for col in X_w03.columns
    if any(word in col.lower()
           for word in ["april", "may", "future"])
]

print("Target/outcome-like feature columns:", target_like)
print("Future-period feature columns:", future_like)

print("\nFinal feature columns:")
print(X_w03.columns.tolist())

print("\nLabel included in features:",
      "label" in X_w03.columns)

print("\nFeature/target overlap:")
print(set(X_w03.columns).intersection({"label"}))

Target/outcome-like feature columns: []
Future-period feature columns: []

Final feature columns:
['content_age_days', 'word_count', 'gsc_impressions_90d', 'gsc_clicks_90d', 'ga4_pageviews_90d']

Label included in features: False

Feature/target overlap:
set()


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


I excluded the future April 2026 click outcome because it is the prediction target and would directly leak the answer into the features. I also excluded `content_hash_id` because it is an identifier rather than a predictive content signal. `visible_query_count` from the earlier feature frame was not included in the final model because the final audited feature set was defined as the five signals used consistently in the modeling notebook.


In [7]:
excluded_features = {
    "label": "Future April outcome; using it as a feature would leak the target.",
    "future_gsc_clicks": "Directly derived from the future outcome period.",
    "content_hash_id": "Identifier, not a page-level predictive signal.",
    "visible_query_count": "Available in an earlier feature frame but excluded from the final five-feature model."
}

print("Excluded fields and reasons:")
for field, reason in excluded_features.items():
    print(f"- {field}: {reason}")

print("\nFinal feature set:")
print(feature_cols)

print("\nExcluded fields present in final features:",
      set(excluded_features).intersection(feature_cols))

Excluded fields and reasons:
- label: Future April outcome; using it as a feature would leak the target.
- future_gsc_clicks: Directly derived from the future outcome period.
- content_hash_id: Identifier, not a page-level predictive signal.
- visible_query_count: Available in an earlier feature frame but excluded from the final five-feature model.

Final feature set:
['content_age_days', 'word_count', 'gsc_impressions_90d', 'gsc_clicks_90d', 'ga4_pageviews_90d']

Excluded fields present in final features: set()


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.